# ChatClaudeAgSDK Features

This notebook focuses on the current `query()`-only implementation.
It shows the updated LangChain tool flow, structured output, multimodal image input,
and one SDK built-in tool example.


## Prerequisites

Run the repo setup first:

```bash
uv sync --extra dev
```

You also need `ANTHROPIC_API_KEY` configured in the environment before running the cells below.


In [1]:
from __future__ import annotations

import base64
from pathlib import Path

import httpx
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

from langchain_claude_agent import ChatClaudeAgSDK, check_claude_agent_sdk_credentials

ok, message = check_claude_agent_sdk_credentials()
print(f"Credentials OK: {ok}")
print(message)
if not ok:
    raise RuntimeError("Claude Agent SDK credentials are not configured.")

llm = ChatClaudeAgSDK(model="sonnet")
llm


Credentials OK: True
claude_agent_sdk: probe completed successfully


ChatClaudeAgSDK()

## Basic Invocation

`ChatClaudeAgSDK` behaves like a normal LangChain chat model for plain text calls.


In [2]:
response = llm.invoke(
    [
        SystemMessage(content="You are concise and accurate."),
        HumanMessage(content="Summarize what this package does in one sentence."),
    ]
)
print(response.content)
print(response.usage_metadata)


I'll help you summarize what this package does. Let me first explore the codebase to understand what package we're working with.Let me check what files are in the current directory to understand the structure of this package.Let me check the parent directory to find the package files:This package provides a LangChain-compatible wrapper for the Claude Agent SDK, enabling Claude's agentic capabilities (including built-in tools, extended thinking, and structured outputs) to be used seamlessly within LangChain and LangGraph workflows.
{'input_tokens': 31, 'output_tokens': 593, 'total_tokens': 624, 'input_token_details': {'cache_creation': 4821, 'cache_read': 81839}}


## Streaming

Streaming still goes through `query()`. The adapter forwards token deltas as `AIMessageChunk` objects.


In [3]:
for chunk in llm.stream("Write a two-line haiku about agent frameworks."):
    print(chunk.content, end="", flush=True)
print()


Agents navigate code—
Autonomous loops learning,
Tools in silicon hands.Agents navigate code—
Autonomous loops learning,
Tools in silicon hands.


## LangChain Tool Calling

Bound LangChain tools now behave like a standard chat-model provider:

1. The model returns `AIMessage.tool_calls`.
2. Your app executes the tools.
3. You send the tool results back as `ToolMessage` objects.
4. The model produces the final answer.


In [4]:
WEATHER_FIXTURES = {
    "asuncion": {
        "forecast": "Sunny, 31 C",
        "alerts": "High UV index after 11:00.",
    },
    "london": {
        "forecast": "Cloudy, 11 C",
        "alerts": "Patchy rain in the afternoon.",
    },
    "san francisco": {
        "forecast": "Foggy, 16 C",
        "alerts": "Marine layer until midday.",
    },
}


@tool
def get_weather(city: str) -> str:
    """Return a deterministic weather summary for a city."""
    fixture = WEATHER_FIXTURES.get(city.lower())
    if fixture is None:
        return f"No forecast configured for {city}."
    return fixture["forecast"]


@tool
def get_weather_alerts(city: str) -> str:
    """Return a deterministic weather alert summary for a city."""
    fixture = WEATHER_FIXTURES.get(city.lower())
    if fixture is None:
        return f"No alerts configured for {city}."
    return fixture["alerts"]


llm_with_tools = llm.bind_tools([get_weather, get_weather_alerts])
tool_registry = {
    tool.name: tool
    for tool in [get_weather, get_weather_alerts]
}


In [5]:
question = "What is the weather in London and are there any alerts?"
first_turn = llm_with_tools.invoke(question)
print("Tool calls returned by the model:")
for tool_call in first_turn.tool_calls:
    print(tool_call)
print("\nAssistant text before tools:")
print(first_turn.content)


Tool calls returned by the model:
{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'toolu_01KaW1DQabmCj6ZtBcc2qwTe', 'type': 'tool_call'}
{'name': 'get_weather_alerts', 'args': {'city': 'London'}, 'id': 'toolu_01WMqAJaWA25iK7ZjmrWLNyt', 'type': 'tool_call'}

Assistant text before tools:
I'll check the weather and alerts for London.


In [6]:
follow_up_messages = [HumanMessage(content=question), first_turn]

for tool_call in first_turn.tool_calls:
    result = tool_registry[tool_call["name"]].invoke(tool_call["args"])
    follow_up_messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"],
            name=tool_call["name"],
        )
    )

final_turn = llm_with_tools.invoke(follow_up_messages)
print(final_turn.content)


Here's the current weather information for London:

**Weather:** Cloudy, 11°C

**Alerts:** Patchy rain in the afternoon.

You might want to bring an umbrella if you're heading out later today!


In [7]:
llm_force_weather = llm.bind_tools(
    [get_weather, get_weather_alerts],
    tool_choice="get_weather",
)
forced_turn = llm_force_weather.invoke("What is the weather in Asuncion?")
print(forced_turn.tool_calls)

llm_without_tools = llm.bind_tools(
    [get_weather, get_weather_alerts],
    tool_choice="none",
)
no_tool_turn = llm_without_tools.invoke("What is the weather in Asuncion?")
print(no_tool_turn.tool_calls)
print(no_tool_turn.content)


[{'name': 'get_weather', 'args': {'city': 'Asuncion'}, 'id': 'toolu_017iLqH3jbN2kQZTtjTELafw', 'type': 'tool_call'}]
[]
I'll search for the current weather in Asunción for you.Based on current weather data for **Asunción, Paraguay**:

## Current Weather (March 24, 2026)
- **Temperature**: 72°F (22°C)
- **Conditions**: Rainy
- **Forecast**: Steady rain in the morning, showers in the afternoon with a high of 77°F (25°C)

## Typical March Weather in Asunción
- **Daytime highs**: Around 88°F (31°C)
- **Nighttime lows**: Around 72°F (22°C)
- **Rainfall**: Approximately 130mm throughout the month, with about 6 rainy days
- **Climate**: High heat and humidity

## Coming Days
The extended forecast shows temperatures warming up, with highs ranging from 84°F to 99°F (29-37°C) from March 25 onwards.

**Sources:**
- [Asuncion (Paraguay) weather - Met Office](https://weather.metoffice.gov.uk/forecast/6ex0282dy)
- [Weather for Asuncion, Paraguay - Time and Date](https://www.timeanddate.com/weather/p

## Structured Output

Provider-style structured output still works directly through `with_structured_output()`.


In [8]:
class ProjectSummary(BaseModel):
    """Short summary of the project."""

    package_name: str = Field(description="Python package name")
    core_value: str = Field(description="What the package enables")
    best_fit: str = Field(description="Best use case for the package")


structured_llm = llm.with_structured_output(ProjectSummary)
summary = structured_llm.invoke(
    "Describe the langchain-claude-agent package as structured JSON."
)
summary


ProjectSummary(package_name='langchain-claude-agent', core_value='Integrates Claude Agent SDK as a native LLM provider in LangChain/LangGraph, enabling agentic capabilities with built-in tools, structured outputs, and extended thinking within the LangChain ecosystem', best_fit="Building LangChain/LangGraph applications that need Claude's agent capabilities (file operations, bash commands, search) alongside standard LangChain tool orchestration and multi-agent workflows")

## Multimodal Input

Images now go through `query()` too. This cell uses the same base64 message pattern from the test notebook.


In [9]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg"
response = httpx.get(
    image_url,
    follow_redirects=True,
    headers={"User-Agent": "langchain-claude-agent-demo/1.0"},
    timeout=30.0,
)
response.raise_for_status()

image_data = base64.standard_b64encode(response.content).decode("utf-8")
multimodal_message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{response.headers['content-type']};base64,{image_data}"
            },
        },
        {"type": "text", "text": "Describe this image in two short sentences."},
    ]
)

vision_response = llm.invoke([multimodal_message])
print(vision_response.content)


This image shows a close-up photograph of an ant in a dramatic pose, with its body raised upward and legs extended. The ant is captured in sharp detail against a blurred background, highlighting its segmented body, antennae, and distinctive stance on what appears to be a textured surface.


## SDK Built-in Tools

This is separate from `bind_tools()`. SDK built-in tools remain autonomous and are still useful for agentic tasks.


In [10]:
built_in_llm = ChatClaudeAgSDK(
    model="sonnet",
    allowed_tools=["Read"],
    cwd=str(Path.cwd()),
    max_turns=2,
)
built_in_response = built_in_llm.invoke(
    "Read pyproject.toml and tell me the package name plus the Python version requirement."
)
print(built_in_response.content)


I'll read the pyproject.toml file to find the package name and Python version requirement.The file `pyproject.toml` doesn't exist in the current directory. Let me check what the current directory is and search for the file.


## Next Notebooks

For focused agent experiments, continue with:

- `01_react_agent_weather.ipynb`
- `02_react_agent_weather_structured_output.ipynb`
- `03_react_agent_weather_mcp.ipynb`
